# 背景扣除后 FITS 的 ASTERIS 时空自监督去噪

这是 ASTERIS 流程的主代码 Notebook，结构与 `noise2noise/01_noise2noise_self_supervised.ipynb`
保持一致：固定帧级切分、预处理验收、训练、推理、科学指标与自动测试依次展开。

ASTERIS4 使用连续 8 帧组成 4 帧输入/4 帧目标；ASTERIS8 使用连续 16 帧组成
8 帧输入/8 帧目标。网络直接复用只读目录中的原作者 3D Restormer-style U-Net，
本项目只提供薄适配层。测试集不参与 checkpoint、输出模式或任何阈值选择。

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import torch
import numpy as np
import pandas as pd
from astropy.io import fits

start = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (start, *start.parents) if (path / "src" / "astr_ir").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("无法定位项目根目录")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

INPUT_ROOT = PROJECT_ROOT / "data" / "processed" / "background"
DATASET_ROOT = PROJECT_ROOT / "data" / "raw" / "our_dataset"
OUTPUT_ROOT = PROJECT_ROOT / "data" / "processed" / "asteris"
FIGURE_ROOT = PROJECT_ROOT / "figures" / "asteris_output"
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

# 安全默认值：执行预处理和小模型 smoke test，但不意外启动长时间训练/推理。
RUN_PREPARE = True
RUN_TRAIN = False
RUN_INFERENCE = False
RUN_CALIBRATION = False
RUN_EVALUATION = False

from astr_ir.asteris.dataset import AsterisPatchDataset, load_registered_stack
from astr_ir.asteris.model import build_asteris_model, upstream_model_path, upstream_source_sha256
from astr_ir.asteris.processor import (
    AsterisConfig,
    calibrate_and_finalize,
    load_calibrated_strength,
    load_manifests,
    prepare_manifests,
    run_inference,
    train_model,
)
from astr_ir.noise2noise.dataset import load_detector_mask

config = AsterisConfig(model="asteris4", patch_t=4, patch_size=64, batch_size=1)
config.validate()
print("device:", "cuda" if torch.cuda.is_available() else "cpu")
print("free disk GB:", round(__import__("shutil").disk_usage(PROJECT_ROOT).free / 2**30, 2))

device: cuda
free disk GB: 178.46


## 1. 原始 ASTERIS 实现与输出语义

In [2]:
source_path = upstream_model_path(config.model)
print("upstream source:", source_path)
print("source SHA-256:", upstream_source_sha256(config.model))
print("output equation: direct_prediction = input + learned_correction")

smoke_model = build_asteris_model(
    "asteris4",
    f_maps=4,
    num_blocks=(1, 1, 1),
    num_refinement_blocks=1,
    heads=(1, 2, 4),
).eval()
smoke_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
smoke_model = smoke_model.to(smoke_device)
with torch.inference_mode():
    smoke_output = smoke_model(torch.randn(1, 1, 4, 16, 16, device=smoke_device))
print("ASTERIS4 smoke output:", tuple(smoke_output.shape))
del smoke_model, smoke_output
if torch.cuda.is_available():
    torch.cuda.empty_cache()

upstream source: D:\Astr_IR\Asteris\ASTERIS_THU-main\asteris\ASTERIS_net_4.py
source SHA-256: 03a8a268aacb69b5de50f3d66d744f8826ca558761f3312886b7ad6513678844
output equation: direct_prediction = input + learned_correction
ASTERIS4 smoke output: (1, 1, 4, 16, 16)


原模型末端已经执行 `output_head(features) + input`，并直接与奇数帧目标计算损失，
因而本项目默认采用直接目标预测。`residual` 模式仅保留作验证集受控实验，不能用测试集选择。

## 2. 先切完整帧，再构造 2T 帧窗口

In [3]:
if RUN_PREPARE:
    split, windows, normalizations = prepare_manifests(
        INPUT_ROOT, DATASET_ROOT, OUTPUT_ROOT, config=config
    )
else:
    split, windows, normalizations = load_manifests(OUTPUT_ROOT)

display(split.groupby(["sequence", "split"]).size().rename("frames").to_frame())
display(windows.groupby(["sequence", "split"])["usable"].agg(["count", "sum"]))
display(pd.DataFrame(normalizations).T)

frame_split = split.set_index("frame_id")["split"]
assert all(
    all(frame_split[frame_id] == row.split for frame_id in row.frame_ids.split("|"))
    for row in windows.itertuples(index=False)
)
assert all(
    not set(row.input_frame_ids.split("|")) & set(row.target_frame_ids.split("|"))
    for row in windows.itertuples(index=False)
)

frames
sequence split             
90000002 guard            4
         test            16
         train           48
         validation      12
90000003 guard            4
         test            16
         train           48
         validation      12

count  sum
sequence split                 
90000002 test            3    3
         train          11   11
         validation      2    2
90000003 test            3    3
         train          11    9
         validation      2    2

,mean,std,fit_low,fit_high,training_sample_count
90000002,5.011965,244.640221,-730.177785,737.669972,2750576.0
90000003,5.181196,243.555433,-726.834526,734.522026,2694812.0


每序列仍使用与 N2N 相同的 48/2/12/2/16 train/guard/validation/guard/test 时间块。
归一化均值和标准差只从训练帧的非源、非盲点、非边缘像元拟合，并写入审计文件。

## 3. 源保护的全局 3σ clipping 与 mask 验收

In [4]:
detector_mask = load_detector_mask(DATASET_ROOT)
sample_window = windows.loc[(windows["split"] == "train") & windows["usable"]].iloc[0]
frame_table = split.set_index("frame_id", drop=False)
sample_rows = frame_table.loc[sample_window.frame_ids.split("|")]
sample_stack, sample_valid, clipping = load_registered_stack(
    sample_rows,
    INPUT_ROOT,
    detector_mask,
    normalizations[str(sample_window.sequence)],
    sigma=config.sigma,
    edge_width=config.edge_width,
    temporal_clip=config.temporal_clip,
)
print({
    "low": clipping.low,
    "high": clipping.high,
    "clipped_fraction": clipping.clipped_fraction,
    "source_pixels": int(clipping.source_mask.sum()),
    "valid_fraction": float(sample_valid.mean()),
})
display(pd.Series(
    clipping.clipping_mask.sum(axis=(1, 2)), name="clipped voxels per frame"
).to_frame())
clipping_audit = pd.read_csv(
    OUTPUT_ROOT / "manifests" / "clipping_audit.csv",
    encoding="utf-8-sig",
    dtype={"sequence": str},
)
display(clipping_audit)
assert clipping_audit["source_quality_gate_passed"].all()
assert not clipping.clipping_mask[:, clipping.source_mask].any()

{'low': -539.2474758446333, 'high': 545.4018089592573, 'clipped_fraction': 0.006622264494414396, 'source_pixels': 1796, 'valid_fraction': 0.9886676073074341}


,clipped voxels per frame
0,5607
1,5701
2,5546
3,5933
4,6039
5,6229
6,6641
7,6847


,sequence,window_id,low,high,clipped_fraction,source_voxels_clipped,source_peak_before,source_peak_after,source_peak_change,source_flux_before,source_flux_after,source_flux_change,source_fwhm_before,source_fwhm_after,source_fwhm_change,source_snr_before,source_snr_after,source_snr_change,source_quality_gate_passed
0,90000002,90000002:train:000-007,-539.247476,545.401809,0.006622,0,891.620047,891.620047,0.0,37751.953125,37751.953125,0.0,27.137961,27.137961,0.0,4.889685,4.889685,0.0,True
1,90000003,90000003:train:000-007,-519.242507,525.270784,0.007223,0,22651.035271,22651.035271,0.0,337156.625000,337156.625000,0.0,9.060858,9.060858,0.0,65.675664,65.675664,0.0,True


时间轴 clipping 默认关闭。短时序中 seeing 与亚像元配准变化会产生真实的时间方向变化；
即使已知目标受 mask 保护，未编目弱源仍可能被误判。只有在验证集峰值、孔径流量、FWHM
和 SNR 质量门均通过后，才应把 `temporal_clip=True` 纳入正式实验。

## 4. 时空 patch、同步增强与 masked loss

In [5]:
dataset = AsterisPatchDataset(
    split, windows, INPUT_ROOT, detector_mask, normalizations,
    split="train", patch_size=config.patch_size, samples_per_epoch=4,
    sigma=config.sigma, edge_width=config.edge_width,
    temporal_clip=config.temporal_clip, augment=True,
)
sample = dataset[0]
display(pd.Series({
    "input shape": tuple(sample["input"].shape),
    "target shape": tuple(sample["target"].shape),
    "loss-mask shape": tuple(sample["loss_mask"].shape),
    "valid loss fraction": float(sample["loss_mask"].mean()),
    "window": sample["window_id"],
}).to_frame("value"))
assert sample["input"].shape == sample["target"].shape == sample["loss_mask"].shape
assert sample["input"].shape[:2] == (1, config.patch_t)

,value
input shape,"(1, 4, 64, 64)"
target shape,"(1, 4, 64, 64)"
loss-mask shape,"(1, 4, 64, 64)"
valid loss fraction,0.973511
window,90000003:train:032-039


## 5. 训练与最佳验证 checkpoint

In [6]:
checkpoint = OUTPUT_ROOT / "checkpoints" / "best_checkpoint.pt"
if RUN_TRAIN:
    checkpoint, history = train_model(
        INPUT_ROOT, DATASET_ROOT, OUTPUT_ROOT, config=config,
        device="cuda" if torch.cuda.is_available() else "cpu",
    )
elif (OUTPUT_ROOT / "checkpoints" / "training_history.csv").exists():
    history = pd.read_csv(OUTPUT_ROOT / "checkpoints" / "training_history.csv", encoding="utf-8-sig")
else:
    history = pd.DataFrame()
    print("尚未训练；把 RUN_TRAIN 改为 True 后执行本单元。")
if not history.empty:
    display(history.loc[history["validation_loss"].idxmin()].to_frame("best"))
    display(history[["epoch", "train_loss", "validation_loss"]].tail(10))

,best
epoch,29.000000
train_loss,0.592574
train_stack_l1,0.332584
train_mean_l2,0.259989
validation_loss,0.547411
validation_stack_l1,0.312277
validation_mean_l2,0.235134


,epoch,train_loss,validation_loss
20,21,0.584342,0.556435
21,22,0.601605,0.554675
22,23,0.566090,0.555371
23,24,0.587651,0.550509
24,25,0.606996,0.551753
25,26,0.591766,0.549443
26,27,0.581387,0.548688
27,28,0.589927,0.549105
28,29,0.592574,0.547411
29,30,0.595497,0.547703


## 6. 原始推理、验证集强度标定与 FITS 科学恒等式

In [7]:
if RUN_INFERENCE:
    raw_statistics = run_inference(
        INPUT_ROOT, DATASET_ROOT, OUTPUT_ROOT, checkpoint,
        config=config, device="cuda" if torch.cuda.is_available() else "cpu",
        overwrite=False,
    )
if RUN_CALIBRATION:
    selected_strength, calibration, statistics = calibrate_and_finalize(
        INPUT_ROOT, DATASET_ROOT, OUTPUT_ROOT, checkpoint,
        config=config, overwrite=False,
    )
elif (OUTPUT_ROOT / "manifests" / "strength_calibration.csv").exists():
    calibration = pd.read_csv(
        OUTPUT_ROOT / "manifests" / "strength_calibration.csv", encoding="utf-8-sig"
    )
    selected_strength = load_calibrated_strength(OUTPUT_ROOT)
else:
    calibration = pd.DataFrame()
    selected_strength = np.nan
if not RUN_CALIBRATION and (OUTPUT_ROOT / "asteris_statistics.csv").exists():
    statistics = pd.read_csv(
        OUTPUT_ROOT / "asteris_statistics.csv", encoding="utf-8-sig", dtype={"sequence": str}
    )
elif not RUN_CALIBRATION:
    statistics = pd.DataFrame()
    print("尚无最终产品；依次启用 RUN_INFERENCE 和 RUN_CALIBRATION。")
if not calibration.empty:
    display(calibration)
    print("validation-selected alpha:", selected_strength)
if not statistics.empty:
    test_statistics = statistics.loc[statistics["split"] == "test"]
    display(test_statistics.groupby("sequence")[
        ["noise_ratio", "photometry_change_fraction", "aperture_snr_ratio"]
    ].median())
    display(test_statistics.groupby("sequence")["photometry_change_fraction"].apply(
        lambda values: 100 * values.abs().max()
    ).rename("max |flux change| [%]").to_frame())
    assert statistics["equation_max_abs_error_float32"].max() == 0

,strength,validation_frames,validation_max_abs_photometry_change,validation_median_noise_ratio,validation_median_snr_ratio,passes_photometry_gate
0,0.00,12,0.000000,1.000000,1.000000,True
1,0.05,12,0.003176,0.950695,1.047041,True
2,0.10,12,0.006236,0.901317,1.108408,True
3,0.15,12,0.008207,0.851922,1.168799,True
4,0.20,12,0.010872,0.802496,1.229254,False
5,0.25,12,0.013700,0.753027,1.307781,False
6,0.30,12,0.017011,0.703680,1.396085,False
7,0.35,12,0.021135,0.654282,1.490098,False
8,0.40,12,0.023252,0.604825,1.626612,False
9,0.45,12,0.026948,0.555559,1.750195,False


validation-selected alpha: 0.15


,noise_ratio,photometry_change_fraction,aperture_snr_ratio
sequence,,,
90000002,0.851877,-0.004564,1.162867
90000003,0.852093,-0.000484,1.166740


,max |flux change| [%]
sequence,
90000002,9.216561
90000003,1.744327


## 7. 与 Noise2Noise 共用的盲伪源评估接口

In [8]:
if RUN_EVALUATION:
    completed = subprocess.run(
        [
            sys.executable, "scripts/run_source_evaluation.py",
            "--model", "asteris", "--device", "cuda" if torch.cuda.is_available() else "cpu",
            "--model-output-root", str(OUTPUT_ROOT),
            "--evaluation-root", str(PROJECT_ROOT / "data" / "processed" / "evaluation" / "asteris"),
        ],
        cwd=PROJECT_ROOT, capture_output=True, text=True, check=False,
    )
    print(completed.stdout)
    if completed.stderr.strip():
        print(completed.stderr)
    if completed.returncode != 0:
        raise RuntimeError("ASTERIS source evaluation failed")
else:
    print("RUN_EVALUATION=False：不会在无 checkpoint 时启动长时间盲评估。")
evaluation_root = PROJECT_ROOT / "data" / "processed" / "evaluation" / "asteris"
if (evaluation_root / "metrics_by_snr.csv").exists():
    blind_metrics = pd.read_csv(evaluation_root / "metrics_by_snr.csv", encoding="utf-8-sig")
    evaluation_summary = pd.read_csv(evaluation_root / "evaluation_summary.csv", encoding="utf-8-sig")
    display(blind_metrics[["method", "target_snr", "completeness", "purity", "f1"]])
    display(evaluation_summary)

RUN_EVALUATION=False：不会在无 checkpoint 时启动长时间盲评估。


,method,target_snr,completeness,purity,f1
0,input,2.0,0.015625,1.000000,0.030769
1,output,2.0,0.031250,0.533333,0.059041
2,input,3.0,0.132812,1.000000,0.234483
3,output,3.0,0.167969,0.877551,0.281967
4,input,4.0,0.367188,1.000000,0.537143
5,output,4.0,0.480469,0.946154,0.637306
6,input,5.0,0.730469,0.994681,0.842342
7,output,5.0,0.804688,0.971698,0.880342
8,input,7.0,0.972656,1.000000,0.986139
9,output,7.0,0.972656,0.976471,0.974560


,metric,value
0,snr_at_50pct_completeness_input,4.365591
1,snr_at_50pct_completeness_output,4.060241
2,snr_limit_improvement_at_50pct_completeness,0.305350
3,snr_at_90pct_completeness_input,6.400000
4,snr_at_90pct_completeness_output,6.134884
5,snr_limit_improvement_at_90pct_completeness,0.265116
6,completeness_gain_snr_2,0.015625
7,completeness_gain_snr_3,0.035156
8,completeness_gain_snr_4,0.113281
9,completeness_gain_snr_5,0.074219


## 8. ASTERIS 专项自动测试

In [9]:
completed = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_asteris.py"],
    cwd=PROJECT_ROOT, capture_output=True, text=True, check=False,
)
print(completed.stdout)
if completed.stderr.strip():
    print(completed.stderr)
if completed.returncode != 0:
    raise RuntimeError("ASTERIS tests failed")

..............                                                           [100%]
14 passed in 5.80s



## 结论

当前 Notebook 已把上游背景扣除 FITS、固定数据隔离、源保护 3σ clipping、训练集归一化、
ASTERIS4/8 原始 3D 网络、masked 自监督损失、双向时序推理、验证集 α 标定和统一科学评估
连接成一条可复现实验链。所有模型与阈值选择只使用训练/验证集，测试集只报告最终结果。